In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Shape: {df_food.shape}")


In [ ]:
# Task 2: Write your code here:
df_food.head(10)

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food = df_food.drop("Order_ID",axis=1)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df_food):
  missing_values = df_food.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nWe need to handle missing values.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_food)

In [ ]:
df_food['Weather'] = df_food['Weather'].fillna('none')
df_food['Traffic_Level'] = df_food['Traffic_Level'].fillna('none')
df_food['Time_of_Day'] = df_food['Time_of_Day'].fillna('none')
df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mode()[0], inplace=True)
df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mode()[0], inplace=True)

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_food.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_food.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")



In [ ]:
from sklearn.preprocessing import OneHotEncoder
# Task 4: Write your code here:
categorical_cols = df_food.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    onehot = OneHotEncoder()
    #  Apply fit_transform to encode the column
    df_food[col] = onehot.fit_transform(df_food[col])

df_food.head()

In [ ]:
# Task 5: Write your code here:

numerical_cols =  df_food.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# Apply fit_transform to scale the numerical columns
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])

df_food.head()

In [ ]:
# Task 6: Write your code here:

plt.figure(figsize=(6, 4))
sns.countplot(data=df_food, x='Delivery_Time')
plt.title('Distribution of Target Variable (Delivery_Time)')
plt.xlabel('Delivery_Time')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df_food.drop("Delivery_Time",axis=1)
y = df_food['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
sklearn_models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
skf = StratifiedKFold(n_splits= 5, shuffle=True, random_state=42)
# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []
n_splits = 5
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model = RandomForestRegressor(n_estimators=100)



    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_losses.append(losses)
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
def pred_y(df_food, y_pred):
  df_food[y_pred].hist(bins=30, edgecolor='black')

  plt.title(f"predicted delivery time ({y_pred})")
  plt.xlabel(delivery time)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

pred_y(df_food, "y_pred")

In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
import numpy as np
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
sklearn_models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
# Storage for results
all_results = {}
n_splits = 5
for name in sklearn_models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}
all_results = {
    "Random Forest Regressor": {"mse": [], "rmse": [], "r2": []},
    "CatBoost": {"mse": [], "rmse": [], "r2": []}}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)


In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")